# NB00: Feature matrix

Builds the full sample × feature matrix combining:
1. CLR-transformed genus-level relative abundances (from MicrobeAtlas OTU data via Spark)
2. Genus-weighted functional features (top-N per functional category + PCA)
3. CWM features (from existing `hmp_feature_matrix.parquet`)
4. Environmental covariates (env + mob from existing `hmp_feature_matrix.parquet`)
5. Metal targets (log1p-transformed GeoROC concentrations)

**Outputs**
- `data/feature_matrix.parquet` — full sample × feature matrix
- `data/coverage_report.csv` — CWM/genus-weighted coverage per sample
- `data/genus_ra.parquet` — raw genus RA (for external validation reuse)

In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path

for _cand in [Path.cwd() / 'scripts', Path.cwd().parent / 'scripts']:
    if _cand.exists():
        sys.path.insert(0, str(_cand))
        break

DATA_DIR = next(p for p in [Path.cwd() / 'data', Path.cwd().parent / 'data'] if p.exists())

try:
    spark
except NameError:
    from berdl_notebook_utils.setup_spark_session import get_spark_session
    spark = get_spark_session()
print('Spark ready:', spark.version)

## 1. Load base feature matrix

Reuse the hybrid_metal_prediction feature matrix as the base: it contains env features, CWM,
and log1p metal targets for 42,037 MicrobeAtlas soil samples. We will join CLR and genus-weighted
features to it by sample_id.

In [ ]:
base = pd.read_parquet(DATA_DIR / 'hmp_feature_matrix.parquet')
print(f'Base feature matrix: {base.shape}')
print('Columns:', base.columns.tolist()[:20], '...')
sample_ids = base.index.tolist()
print(f'Sample IDs: {len(sample_ids):,}')

## 2. Genus RA from Spark

In [ ]:
from cwm_utils import load_otu_bridge, load_genus_ra_from_spark

GENUS_RA_PATH = DATA_DIR / 'genus_ra.parquet'

if GENUS_RA_PATH.exists():
    print(f'Loading cached genus RA from {GENUS_RA_PATH}')
    genus_ra_wide = pd.read_parquet(GENUS_RA_PATH)
    print(f'Genus RA matrix: {genus_ra_wide.shape[0]:,} samples × {genus_ra_wide.shape[1]:,} genera')
else:
    bridge = load_otu_bridge(DATA_DIR / 'otu_pangenome_link_v2.csv')
    print(f'OTU bridge: {len(bridge):,} OTUs → {bridge["genus_lower"].nunique():,} genera')

    # Chunked loading: 1000 samples/chunk avoids both the huge SQL string (42k IDs → kernel OOM)
    # and the 1GB toPandas() limit. Each chunk returns ~25 MB.
    CHUNK_SIZE = 1000
    n_chunks = (len(sample_ids) + CHUNK_SIZE - 1) // CHUNK_SIZE
    print(f'Loading genus RA in {n_chunks} chunks of up to {CHUNK_SIZE} samples...')

    chunks = []
    for i in range(0, len(sample_ids), CHUNK_SIZE):
        chunk_ids = sample_ids[i:i + CHUNK_SIZE]
        try:
            chunk_ra = load_genus_ra_from_spark(spark, sample_ids=chunk_ids, otu_bridge=bridge)
            chunks.append(chunk_ra)
        except ValueError as e:
            print(f'  WARNING chunk {i//CHUNK_SIZE+1}/{n_chunks}: {e} — skipping')
        if (i // CHUNK_SIZE + 1) % 10 == 0 or i + CHUNK_SIZE >= len(sample_ids):
            print(f'  Chunk {i//CHUNK_SIZE+1}/{n_chunks}: {chunk_ra.shape}')

    print(f'\nLoaded {len(chunks)} chunks. Concatenating...')
    genus_ra_wide = pd.concat(chunks).fillna(0.0)
    genus_ra_wide = genus_ra_wide.groupby(level=0).first()
    print(f'Genus RA matrix: {genus_ra_wide.shape[0]:,} samples × {genus_ra_wide.shape[1]:,} genera')

    genus_ra_wide.to_parquet(GENUS_RA_PATH)
    print('Saved genus_ra.parquet')


## 3. CLR transform

Select top-200 genera by mean RA before CLR to keep dimensionality tractable for XGBoost.
CLR is computed on the selected genera (pseudocount 1e-6).

In [ ]:
from composition_utils import clr_transform, select_top_genera

TOP_N_GENERA = 200

genus_ra_top = select_top_genera(genus_ra_wide, top_n=TOP_N_GENERA, by='mean_ra')
print(f'Selected top-{TOP_N_GENERA} genera: {genus_ra_top.shape}')

clr_feats = clr_transform(genus_ra_top)
print(f'CLR features: {clr_feats.shape}')
print('Sample CLR stats:', clr_feats.describe().loc[['mean', 'std']].round(3))

## 4. Genus-weighted functional features

For each functional category in `genus_trait_table.csv`, compute RA_g × density_gk for each genus g.
Retain the top-20 genera per category by mean contribution, plus 10 PCA components of the full
genus × category contribution matrix.

In [ ]:
from composition_utils import compute_genus_weighted_features
from cwm_utils import load_genus_densities

densities = load_genus_densities(genus_trait_path=DATA_DIR / 'genus_trait_table.csv')
print(f'Density table: {densities.shape} ({densities.columns.tolist()})')

TOP_N_PER_CAT = 20
N_PCA = 10

# Use full genus_ra_wide (not top-200) so rare genera with high density aren't excluded
gw_feats = compute_genus_weighted_features(
    genus_ra_wide, densities,
    top_n_per_cat=TOP_N_PER_CAT,
    n_pca=N_PCA,
)
print(f'Genus-weighted features: {gw_feats.shape}')
print('Feature name examples:', gw_feats.columns[:5].tolist(), '...')

## 5. Coverage report

In [ ]:
from composition_utils import cwm_coverage_fraction

coverage = cwm_coverage_fraction(genus_ra_wide, densities)
coverage_df = coverage.to_frame()
coverage_df['low_coverage'] = coverage_df['coverage_fraction'] < 0.70
coverage_df['clr_genus_count'] = (genus_ra_top > 0).sum(axis=1)

print(f'Coverage: {coverage_df["coverage_fraction"].mean():.3f} mean; '
      f'{coverage_df["low_coverage"].mean():.1%} flagged (<70%)')

coverage_df.to_csv(DATA_DIR / 'coverage_report.csv')
print('Saved coverage_report.csv')


## 6. Assemble and save feature matrix

In [ ]:
# Join CLR and GW features to the base (which has env + CWM + targets)
common_idx = base.index.intersection(clr_feats.index)
print(f'Common samples (base ∩ genus RA): {len(common_idx):,}')

feature_matrix = (
    base.loc[common_idx]
    .join(clr_feats.loc[common_idx], how='left')
    .join(gw_feats.loc[common_idx], how='left')
)

print(f'Final feature matrix: {feature_matrix.shape}')
print('Column groups:')
clr_cols = [c for c in feature_matrix.columns if c.startswith('clr_')]
gw_cols  = [c for c in feature_matrix.columns if c.startswith('gw_')]
print(f'  CLR: {len(clr_cols)}')
print(f'  GW:  {len(gw_cols)}')
print(f'  Remaining: {feature_matrix.shape[1] - len(clr_cols) - len(gw_cols)}')

feature_matrix.to_parquet(DATA_DIR / 'feature_matrix.parquet')
print('Saved feature_matrix.parquet')